In [ ]:

import importlib
import os

import dotenv
import langchain_deepseek
import langchain.chains
import langchain.prompts

dotenv.load_dotenv()

In [ ]:

import src.prompts.code_analysis_prompt
import src.utils.code_validation

importlib.reload(src.prompts.code_analysis_prompt)
importlib.reload(src.utils.code_validation)


In [ ]:
example_snippet = \
"""
from sys import stdin
s = [ord(i) - 97 for i in stdin.readline().strip()]
s1 = [ord(i) - 97 for i in stdin.readline().strip()]
n = len(s)
m = len(s1)
ans = 0
mod = 10 ** 9 + 7
dp = [[0 for i in range(n)] for j in range(26)]
for i in range(m):
	arr = [0 for j in range(n)]
	for j in range(n):
		if s1[i] == s[j]:
			arr[j] = 1
			if j > 0:
				arr[j] = (arr[j] + dp[s[j - 1]][j - 1]) % mod
	for j in range(n):
		dp[s1[i]][j] = (arr[j] + dp[s1[i]][j]) % mod
x = 0
for i in dp:
	x = (x + sum(i)) % mod
print(x)
"""

In [ ]:
llm = langchain_deepseek.ChatDeepSeek(
    model="deepseek-chat",
    temperature=0.0,  # recommended setting for coding/math
    max_tokens=1024,
    timeout=None,
    max_retries=3,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
)

In [ ]:

code_analysis_chain = langchain.chains.LLMChain(
    llm=llm,
    prompt=src.prompts.code_analysis_prompt.code_analysis_prompt,
    verbose=True,
    output_parser=src.prompts.code_analysis_prompt.code_analysis_output_parser,
)

def analyze_code(code_string):
    src.utils.code_validation.validate_code(code_string)
    return code_analysis_chain.run({"code": code_string})

response = analyze_code(example_snippet)

In [ ]:
print(response.model_dump_json(indent=2))